## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

## 2. Load Cleaned Datasets

In [2]:
# ------------------------------------------------------------
# These datasets have already passed the cleaning and
# validation stage and are ready for transformation.
# ------------------------------------------------------------

farmer_df = pd.read_csv("C:/Users/HP PC/Documents/jacob_git_projects/Monitoring-and-Evaluation-System/dataset/02_cleaned/"
                        "farmer_registration_clean.csv")

monitoring_df = pd.read_csv("C:/Users/HP PC/Documents/jacob_git_projects/Monitoring-and-Evaluation-System/dataset/02_cleaned/"
                            "farm_monitoring_visit_clean.csv")

training_df = pd.read_csv("C:/Users/HP PC/Documents/jacob_git_projects/Monitoring-and-Evaluation-System/dataset/02_cleaned/"
                          "training_attendance_clean.csv")

input_df = pd.read_csv("C:/Users/HP PC/Documents/jacob_git_projects/Monitoring-and-Evaluation-System/dataset/02_cleaned/"
                       "input_distribution_clean.csv")

## 3. Convert Date Columns

In [3]:
# ------------------------------------------------------------
# Convert date and time fields to datetime objects to
# support duration and age calculations.
# ------------------------------------------------------------

farmer_df["form.section_1_farmer_identification.date_of_birth"] = pd.to_datetime(farmer_df["form.section_1_farmer_identification.date_of_birth"])

farmer_df["started_time"] = pd.to_datetime(farmer_df["started_time"])

farmer_df["completed_time"] = pd.to_datetime(farmer_df["completed_time"])

monitoring_df["started_time"] = pd.to_datetime(monitoring_df["started_time"])

monitoring_df["completed_time"] = pd.to_datetime(monitoring_df["completed_time"])

training_df["started_time"] = pd.to_datetime(training_df["started_time"])

training_df["completed_time"] = pd.to_datetime(training_df["completed_time"])

input_df["started_time"] = pd.to_datetime(input_df["started_time"])

input_df["completed_time"] = pd.to_datetime(input_df["completed_time"])

## 4. Create Working Copies

In [4]:
# ------------------------------------------------------------
# Preserve the cleaned datasets by creating transformed
# copies that will receive new derived columns.
# This preserves lineage and makes debugging straightforward.
# ------------------------------------------------------------

farmer_transformed = farmer_df.copy()

monitoring_transformed = monitoring_df.copy()

training_transformed = training_df.copy()

input_transformed = input_df.copy()

## 5. Farmer Age

In [5]:
# ------------------------------------------------------------
# Calculate each farmer's age in years based on their
# date of birth.
# ------------------------------------------------------------

today = pd.Timestamp.today()

farmer_transformed["farmer_age"] = ((today - farmer_transformed["form.section_1_farmer_identification.date_of_birth"]).dt.days // 365)

## 6. Registration Form Duration

In [6]:
# ------------------------------------------------------------
# Calculate the number of minutes taken to complete
# each farmer registration form.
# ------------------------------------------------------------

farmer_transformed["registration_duration_minutes"] = (farmer_transformed["completed_time"] - farmer_transformed["started_time"]
                                                      ).dt.total_seconds() / 60


## 7. Registration GPS Coordinates

In [7]:
# ------------------------------------------------------------
# Split the CommCare GPS field into separate latitude
# and longitude columns for spatial analysis.
# ------------------------------------------------------------

registration_gps = farmer_transformed["form.section_6_farm_location.farm_gps_location"].str.split(expand=True)

farmer_transformed["registration_latitude"] = registration_gps[0].astype(float)

farmer_transformed["registration_longitude"] = registration_gps[1].astype(float)

## 8. Preview Farmer Registration Transformations

In [8]:
farmer_transformed[
    [
        "farmer_age",
        "registration_duration_minutes",
        "registration_latitude",
        "registration_longitude"
    ]
].head()

,farmer_age,registration_duration_minutes,registration_latitude,registration_longitude
0,32,7.0,10.343592,7.337582
1,39,8.0,10.537141,7.510400
2,45,7.0,10.387220,7.300566
3,41,7.0,10.223170,7.551204
4,25,6.0,10.394166,7.415694


## 9. Monitoring Form Duration

In [9]:
# ------------------------------------------------------------
# Calculate the number of minutes taken to complete
# each farm monitoring visit form.
# ------------------------------------------------------------

monitoring_transformed["monitoring_duration_minutes"] = (monitoring_transformed["completed_time"] - monitoring_transformed["started_time"]
                                                        ).dt.total_seconds() / 60


## 10. Monitoring Visit GPS Coordinates

In [10]:
# ------------------------------------------------------------
# Split the monitoring visit GPS field into separate
# latitude and longitude columns.
# ------------------------------------------------------------

monitoring_gps = monitoring_transformed["form.section_5_farm_location.capture_monitoring_visit_gps"].str.split(expand=True)

monitoring_transformed["monitoring_latitude"] = monitoring_gps[0].astype(float)

monitoring_transformed["monitoring_longitude"] = monitoring_gps[1].astype(float)


## 11. Bring Registration GPS into Monitoring Dataset

In [11]:
# ------------------------------------------------------------
# Join the farmer registration dataset using the
# farmer case ID so that registration and monitoring
# GPS coordinates can be compared.
# ------------------------------------------------------------

registration_locations = farmer_transformed[["form.case.@case_id", "registration_latitude", "registration_longitude"]]

monitoring_transformed = monitoring_transformed.merge(

    registration_locations,

    on="form.case.@case_id",

    how="left"

)

## 12. Haversine Distance Function

In [12]:
# ------------------------------------------------------------
# Calculate the distance between two GPS coordinates
# in metres.
# ------------------------------------------------------------

def haversine_distance(
    lat1,
    lon1,
    lat2,
    lon2
):

    R = 6371000

    lat1 = radians(lat1)
    lon1 = radians(lon1)

    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        +
        cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

    return R * c

## 13. GPS Distance from Registration Location

In [13]:
# ------------------------------------------------------------
# Calculate the distance between the farmer's
# registration GPS and monitoring visit GPS.
# ------------------------------------------------------------

monitoring_transformed["gps_distance_meters"] = monitoring_transformed.apply(

    lambda row:

    haversine_distance(

        row["registration_latitude"],
        row["registration_longitude"],
        row["monitoring_latitude"],
        row["monitoring_longitude"]

    ),

    axis=1

)

## 14. GPS Validation Flag

In [14]:
# ------------------------------------------------------------
# Monitoring visits within 100 metres of the
# registration location are considered valid.
# ------------------------------------------------------------

monitoring_transformed["gps_within_expected_range"] = np.where(monitoring_transformed["gps_distance_meters"] <= 100, "Yes", "No")


## 15. Preview Monitoring Transformations

In [15]:
# ------------------------------------------------------------

monitoring_transformed[
    [
        "monitoring_duration_minutes",
        "registration_latitude",
        "registration_longitude",
        "monitoring_latitude",
        "monitoring_longitude",
        "gps_distance_meters",
        "gps_within_expected_range"
    ]
].head()

,monitoring_duration_minutes,registration_latitude,registration_longitude,monitoring_latitude,monitoring_longitude,gps_distance_meters,gps_within_expected_range
0,6.0,10.343592,7.337582,10.343875,7.337217,50.902067,Yes
1,3.0,10.343592,7.337582,10.343899,7.338025,59.256394,Yes
2,6.0,10.537141,7.510400,10.537313,7.510863,54.150415,Yes
3,6.0,10.387220,7.300566,10.387539,7.300917,52.323603,Yes
4,3.0,10.387220,7.300566,10.387397,7.301058,57.377183,Yes


## 16. Training Attendance Form Duration

In [16]:
# ------------------------------------------------------------
# Calculate the number of minutes taken to complete
# each training attendance form.
# ------------------------------------------------------------

training_transformed["training_duration_minutes"] = (training_transformed["completed_time"] - training_transformed["started_time"]
                                                    ).dt.total_seconds() / 60


## 17. Preview Training Attendance Transformations

In [17]:
training_transformed[["training_duration_minutes"]].head()

,training_duration_minutes
0,4.0
1,2.0
2,5.0
3,3.0
4,4.0


## 18. Input Distribution Form Duration

In [18]:
# ------------------------------------------------------------
# Calculate the number of minutes taken to complete
# each input distribution form.
# ------------------------------------------------------------

input_transformed["distribution_duration_minutes"] = (input_transformed["completed_time"] - input_transformed["started_time"]
                                                     ).dt.total_seconds() / 60


## 19. Preview Input Distribution Transformations

In [19]:
input_transformed[["distribution_duration_minutes"]].head()

,distribution_duration_minutes
0,8.0
1,3.0
2,3.0
3,7.0
4,5.0


## 20. Export Transformed Datasets

In [20]:
# ------------------------------------------------------------
# Export the transformed datasets for SQL modelling
# and downstream reporting.
# ------------------------------------------------------------

farmer_transformed.to_csv("C:/Users/HP PC/Documents/jacob_git_projects/Monitoring-and-Evaluation-System/dataset/03_transformed/"
                          "farmer_registration_transformed.csv",
                          index=False)

monitoring_transformed.to_csv("C:/Users/HP PC/Documents/jacob_git_projects/Monitoring-and-Evaluation-System/dataset/03_transformed/"
                              "farm_monitoring_visit_transformed.csv",
                              index=False)

training_transformed.to_csv("C:/Users/HP PC/Documents/jacob_git_projects/Monitoring-and-Evaluation-System/dataset/03_transformed/"
                            "training_attendance_transformed.csv",
                            index=False)

input_transformed.to_csv("C:/Users/HP PC/Documents/jacob_git_projects/Monitoring-and-Evaluation-System/dataset/03_transformed/"
                         "input_distribution_transformed.csv",
                         index=False)

print("All transformed datasets exported successfully.")


All transformed datasets exported successfully.
